# Hangman — n-gram baseline

A statistical floor to get a valid submission on the board and to prove the
end-to-end pipeline. Self-contained, CPU only, no internet, ~12 minutes.

For each hidden slot the model asks what letter usually sits between the known
characters around it, backing off from four-character contexts to one when the
long ones are unseen. Slot distributions are combined across the word by
noisy-OR. It never needs to have seen the word — only its neighbourhoods —
which is what matters here, because **no test word appears in `train.txt`**.

Measured: **50.799%** win rate, 4.567 mean strikes.


In [ ]:
# ============================================================================
# Hangman baseline - character n-gram back-off with noisy-OR aggregation.
#
# Self-contained: no attached datasets, no internet, CPU only, ~12 minutes.
# This is the statistical floor the neural model has to beat; it is kept in the
# repo as an ensemble member and as a sanity check on the simulator.
# ============================================================================
import csv, time
from collections import defaultdict
import numpy as np

def find_competition_dir(names=("train.txt", "test.txt")):
    """Kaggle mounts competition data at different paths on different kernels."""
    import os
    for root in ("/kaggle/input", "/kaggle/input/competitions", "data", "."):
        if not os.path.isdir(root):
            continue
        if all(os.path.exists(os.path.join(root, n)) for n in names):
            return root
        for child in sorted(os.listdir(root)):
            d = os.path.join(root, child)
            if os.path.isdir(d) and all(os.path.exists(os.path.join(d, n)) for n in names):
                return d
    raise FileNotFoundError("train.txt / test.txt not found under /kaggle/input")


COMP = find_competition_dir()
print("data:", COMP)
ORDERS = (4, 3, 2, 1)
ALPHA = 0.4          # back-off weight kept on the shorter context
MAX_LIVES = 6
BOUNDARY, UNKNOWN = 26, 27


def load_words(path):
    with open(path, encoding="utf-8") as fh:
        return [w for w in (line.strip().lower() for line in fh) if w]


class NGramModel:
    """P(letter at a slot | the known characters around it), with back-off."""

    def __init__(self, words, orders=ORDERS, alpha=ALPHA):
        self.orders, self.alpha = tuple(sorted(orders, reverse=True)), alpha
        pad = max(self.orders)
        left = defaultdict(lambda: np.zeros(26, dtype=np.float32))
        right = defaultdict(lambda: np.zeros(26, dtype=np.float32))
        prior = np.zeros(26)

        for word in words:
            codes = [ord(c) - 97 if "a" <= c <= "z" else -1 for c in word]
            padded = [BOUNDARY] * pad + codes + [BOUNDARY] * pad
            for i, code in enumerate(codes):
                if code < 0:
                    continue
                prior[code] += 1
                p = pad + i
                for n in self.orders:
                    lk, rk = padded[p - n:p], padded[p + 1:p + 1 + n]
                    if all(0 <= c <= BOUNDARY for c in lk):
                        left[bytes([n]) + bytes(lk)][code] += 1
                    if all(0 <= c <= BOUNDARY for c in rk):
                        right[bytes([n]) + bytes(rk)][code] += 1

        self.left = {k: v / v.sum() for k, v in left.items() if v.sum() > 3}
        self.right = {k: v / v.sum() for k, v in right.items() if v.sum() > 3}
        self.prior = prior / prior.sum()

    def slot(self, padded, p):
        dist = self.prior
        for table, lo, hi in ((self.left, -1, 0), (self.right, 1, 1)):
            for n in self.orders:
                window = padded[p - n:p] if lo < 0 else padded[p + 1:p + 1 + n]
                if any(c == UNKNOWN for c in window):
                    continue
                hit = table.get(bytes([n]) + bytes(window))
                if hit is not None:
                    dist = self.alpha * dist + (1 - self.alpha) * hit
                    break
        return dist

    def scores(self, board, absent):
        """board: list of codes with UNKNOWN for hidden slots. -> (26,) scores."""
        pad = max(self.orders)
        padded = [BOUNDARY] * pad + board + [BOUNDARY] * pad
        log_none = np.zeros(26)
        for j, code in enumerate(board):
            if code == UNKNOWN:
                d = self.slot(padded, pad + j)
                log_none += np.log1p(-np.clip(d, 0, 1 - 1e-9))
        out = -np.expm1(log_none)      # P(letter appears in some hidden slot)
        out[list(absent)] = -1.0
        return out


def play(word, model):
    """One honest game: the model only ever sees the board and the misses."""
    codes = [ord(c) - 97 for c in word]
    present = set(codes)
    revealed, guessed, absent, wrong, seq = set(), set(), set(), 0, []

    while wrong < MAX_LIVES and revealed != present:
        board = [c if c in revealed else UNKNOWN for c in codes]
        s = model.scores(board, absent)
        s[list(guessed)] = -np.inf
        letter = int(s.argmax())
        guessed.add(letter)
        seq.append(chr(97 + letter))
        if letter in present:
            revealed.add(letter)
        else:
            absent.add(letter)
            wrong += 1
    return "".join(seq), revealed == present, wrong


train_words = load_words(f"{COMP}/train.txt")
test_words = load_words(f"{COMP}/test.txt")
print(f"{len(train_words):,} train / {len(test_words):,} test")
print("test words also in train:", len(set(test_words) & set(train_words)))

t = time.time()
model = NGramModel(train_words)
print(f"model built in {time.time()-t:.0f}s "
      f"({len(model.left)+len(model.right):,} contexts)")

t = time.time()
rows, wins, strikes = [], 0, 0
for i, word in enumerate(test_words):
    seq, won, wrong = play(word, model)
    rows.append((i, seq))
    wins += won
    strikes += wrong
    if (i + 1) % 25000 == 0:
        print(f"  {i+1:,}  win rate {100*wins/(i+1):.2f}%  [{time.time()-t:.0f}s]")

print(f"\nwin rate  : {100*wins/len(test_words):.3f}%")
print(f"strikes   : {strikes:,} (mean {strikes/len(test_words):.3f})")

with open("submission.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["word_id", "guessed_letters_string"])
    w.writerows(rows)

# Guardrails against the mistakes that silently cost points.
assert len(rows) == 250_000
for i, seq in rows:
    assert seq.isalpha() and len(set(seq)) == len(seq)
print("submission.csv written and validated")